<a href="https://colab.research.google.com/github/e23189uop/Statistical-Learning-e23189/blob/main/assignment%2307b_e23189/assignment07b_e23189.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Question 1: Bayesian Estimation of a User Ability Parameter

**1. Sequential Likelihood Contribution**
The likelihood contribution of a single new response $y_k \in \{0,1\}$ at step $k$, given the latent ability $\theta$, is the Bernoulli probability:


$$L(y_k \mid \theta) = p_k(\theta)^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$


The joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of independent Bernoulli trials:


$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} p_i(\theta)^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

**2. Mathematical Formulation of the Running Update**
Using Bayes' Theorem, the recursive relationship for the posterior density at step $k$ (ignoring the marginal probability of data as the proportionality constant) is:


$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \times L(y_k \mid \theta)$$

**3. Dynamic Shifting**
If a user correctly answers an item with a high difficulty parameter ($b_k$), the likelihood function $L(y_k=1 \mid \theta) = p_k(\theta)$ will only have high mass at higher values of $\theta$. When this likelihood is multiplied against the prior density, it significantly shifts the mass (and the peak) of the new running posterior density to the right, indicating a strong positive update to the user's estimated ability.

**4. Tracking Certainty and Sharpness**
The discrimination parameter $a_k$ controls the steepness of the 2PL curve.

* A **very large $a_k$** acts like a sharp step function. Answering correctly heavily penalizes low $\theta$ values and greatly reduces the variance (increasing "sharpness") of the posterior distribution.
* A **very small $a_k$** creates a flat probability curve, meaning the item provides very little information. The likelihood is relatively uniform, leaving the posterior variance largely unchanged.

**5. Numerical Implementation of a Running Grid**
To implement this algorithmically:

1. Define a dense, fixed discrete grid of $\theta$ values (e.g., from -4 to 4).
2. Initialize an array representing the prior PDF evaluated at each grid point.
3. For each observed $y_k$, calculate the likelihood array $L(y_k \mid \theta)$ across all grid points.
4. Multiply the prior array by the likelihood array element-wise to get the unnormalized posterior.
5. **Sequential Normalization:** Integrate the unnormalized array using the trapezoidal rule (`np.trapezoid`) over the $\theta$ grid to find the area. Divide the unnormalized posterior array by this area so it sums to a valid probability density of 1.

**6. Python Script (Tasks 1 & 7 combined)**




In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# --- Task 1: Visualizing the Mechanics ---
theta_grid = np.linspace(-4, 4, 1000)
def p_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

fig1 = go.Figure()
for b in [-1.5, 0, 1.5]:
    fig1.add_trace(go.Scatter(x=theta_grid, y=p_correct(theta_grid, a=1.0, b=b), name=f"a=1.0, b={b}"))
fig1.add_trace(go.Scatter(x=theta_grid, y=p_correct(theta_grid, a=2.5, b=0), line=dict(dash='dash'), name="a=2.5, b=0"))
fig1.update_layout(title="2PL IRT Model Mechanics", xaxis_title="Theta", yaxis_title="P(Y=1)")
fig1.show()

# --- Task 7: Evaluating Convergence ---
np.random.seed(42)
n_items = 20
theta_true = 0.75
theta_range = np.linspace(-4, 4, 1000)
posterior = norm.pdf(theta_range, 0, 1)

history_bayes, history_map = [], []

for k in range(n_items):
    a_k = np.random.uniform(0.5, 2.0)
    b_k = np.random.normal(0, 1)

    # Simulate response
    p_true = p_correct(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # Update grid
    likelihood = p_correct(theta_range, a_k, b_k) if y_k == 1 else (1 - p_correct(theta_range, a_k, b_k))
    unnormalized_post = posterior * likelihood
    posterior = unnormalized_post / np.trapezoid(unnormalized_post, theta_range)

    # Track Estimators
    bayes_est = np.trapezoid(theta_range * posterior, theta_range)
    map_est = theta_range[np.argmax(posterior)]

    history_bayes.append(bayes_est)
    history_map.append(map_est)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=history_bayes, mode='lines+markers', name='Bayes Estimator (Mean)'))
fig2.add_trace(go.Scatter(y=history_map, mode='lines+markers', name='MAP Estimator (Mode)'))
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Theta (0.75)")
fig2.update_layout(title="Estimator Convergence over 20 Items", xaxis_title="Item Step (k)", yaxis_title="Theta Estimate")
fig2.show()



*Analysis:* As $k$ increases, the distance between the estimators and $\theta_{\text{true}}$ generally decreases, although it may fluctuate depending on the difficulty of the items. This implies the platform's confidence in its measurement is growing as evidence accumulates.

---

## Question 2: Bayesian Tracking of CTR via Conjugate Beta-Binomial Updates

**1. Sequential Likelihood and Joint History**
The likelihood of a single isolated response is a Bernoulli PMF:


$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$


The joint likelihood for the running history vector is:


$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

**2. Closed-Form Analytical Updates (Conjugacy)**
Using Bayes' Theorem:


$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \times L(y_k \mid \theta)$$


Substituting the Beta prior and Bernoulli likelihood:


$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{\alpha_{k-1}-1} (1-\theta)^{\beta_{k-1}-1} \right] \times \left[ \theta^{y_k} (1-\theta)^{1-y_k} \right]$$

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1-\theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$


This matches the structural kernel of a Beta distribution, proving conjugacy. The updates are:


$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$


The Posterior Mean at step $k$ is:


$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

**3. Dynamic Shifting Mechanics**
An observed click ($y_k = 1$) increments $\alpha_k$ by 1, pushing the peak of the Beta distribution to the right (higher CTR). A non-click ($y_k = 0$) increments $\beta_k$ by 1, pushing the peak leftward. Unlike the 2PL IRT model where the posterior has no standard algebraic form (requiring numerical integration on a grid), the Beta-Binomial conjugacy allows the entire density to be represented strictly by tracking two integers in closed form.

**4. Running Point Estimators**

* **Running Posterior Mean:** $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$

* **Running Maximum A Posteriori (Mode):** $\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$ (for $\alpha_k, \beta_k > 1$)



**5. Performance Tracking Python Script**



In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# --- Task 1: Structural Probability ---
theta_grid = np.linspace(0, 1, 500)
fig1 = go.Figure()
params = [(1,1), (2,8), (8,2)]
for a, b in params:
    fig1.add_trace(go.Scatter(x=theta_grid, y=beta.pdf(theta_grid, a, b), name=f"Beta({a},{b})"))
fig1.update_layout(title="Beta Distribution Priors", xaxis_title="Theta", yaxis_title="Density")
fig1.show()

# --- Task 6: Convergence Analysis ---
np.random.seed(100)
n_impressions = 100
theta_true = 0.35
alpha_k, beta_k = 1, 1

history_bayes, history_map = [], []

for k in range(1, n_impressions + 1):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha_k += y_k
    beta_k += (1 - y_k)

    history_bayes.append(alpha_k / (alpha_k + beta_k))
    # Map estimator logic requires checking bounds
    if alpha_k > 1 and beta_k > 1:
        history_map.append((alpha_k - 1) / (alpha_k + beta_k - 2))
    else:
        history_map.append(alpha_k / (alpha_k + beta_k)) # Fallback if mode undefined

fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=history_bayes, mode='lines', name='Bayes Estimator'))
fig2.add_trace(go.Scatter(y=history_map, mode='lines', name='MAP Estimator'))
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True CTR (0.35)")
fig2.update_layout(title="CTR Estimator Convergence", xaxis_title="Impressions (k)")
fig2.show()



---

## Question 3: Bayesian Estimations for Structural Health Monitoring

**1. Prior Belief Boundaries**
The expected prior stiffness efficiency for $\Theta \sim \text{Beta}(8, 1.5)$ is:


$$\mathbb{E}[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842$$


This distribution is highly left-skewed, concentrating the probability mass near $1.0$, which physically aligns with the assumption that a new or recently inspected component is highly likely to be healthy and fully stiff.

**2. Structural Likelihood Formulation**
Taking the natural logarithm of the physics model gives $\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k$. Since $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, the measurement $y_k$ follows a log-normal distribution. The likelihood of a single continuous sensor measurement is:


$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( - \frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$


The joint likelihood is the product over all $k$ steps:


$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( - \frac{(\ln y_i - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

**3. Non-Conjugate Grid Update**
An exact closed-form solution does not exist because the Beta prior (a polynomial fraction of $\theta$) and the Log-Normal likelihood (an exponential function of $\ln(\theta)$) do not share the same functional algebra; multiplying them together does not produce a known standard distribution family. The recursive relationship is:


$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \times \exp\left( - \frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

**4. Running Point Estimates**
Because there is no closed form, we evaluate these via definite integrals over the domain $(0, 1]$:

* **Running Posterior Mean:** $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$

* **Running MAP:** $\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname{arg\,max}_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$


**5. Grid Approximation and Normalization**
Create a discrete array of $\theta$ points closely bounding the interval, e.g., `np.linspace(0.01, 1.0, 1000)` to avoid $\ln(0)$ boundary issues. On observing $y_k$, evaluate the log-normal PDF equation above for every $\theta$ in the array. Multiply this likelihood array element-wise by the stored posterior array from step $k-1$. Normalize the resulting array by dividing it by its numerical integral: `np.trapezoid(unnormalized_array, theta_grid)`.

---

## Question 4: Gaussian Mixture Clustering as Conditional Updating

**1. Deriving the Marginal Density**
Using the Law of Total Probability:


$$p(x_i) = \sum_{k=1}^K P(X_i=x_i \mid C_i=k) P(C_i=k)$$


Substituting the components yields:


$$p(x_i) = \sum_{k=1}^K \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \phi_k = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$


This is called a Gaussian mixture density because the total probability density over the space is formulated as a linear combination (a "mixture") of individual Gaussian densities, weighted by their prior mixing proportions ($\phi_k$).

**2. Deriving the Posterior Cluster Probability**
By direct application of Bayes' rule and substitution:


$$P(C_i=k \mid X_i=x_i) = \frac{P(X_i=x_i \mid C_i=k)P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i \mid C_i=j)P(C_i=j)} = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$


This responsibility $\gamma_{ik}$ is interpreted as the posterior probability because it represents our updated belief that data point $x_i$ was generated by cluster $k$, given its spatial location and the prior probabilities.

**3. One-Hot Encoding Expectation**
Since $Z_{ik}$ is an indicator variable (taking values $1$ or $0$), its expectation is the probability that it equals $1$:


$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = 1 \cdot P(C_i=k \mid X_i=x_i) + 0 \cdot P(C_i \neq k \mid X_i=x_i) = \gamma_{ik}$$


Therefore, the expected vector is simply the vector of responsibilities:


$$\mathbb{E}[Z_i \mid X_i=x_i] = [\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$$


This proves that the soft cluster assignment is strictly the mathematical conditional expectation of the latent state.

**4. Soft vs. Hard Clustering**
Soft clustering assigns a probability distribution over all clusters for every point (e.g., $x_i$ is 80% Cluster A, 20% Cluster B). Hard clustering forces a deterministic binary decision, assigning $x_i$ entirely to the cluster with the highest responsibility (100% Cluster A, 0% Cluster B).

**5. Conditional Expectation of the Observation**
Given $X_i \mid C_i=k \sim \mathscr{N}(\mu_k, \Sigma_k)$, the expected value of a normal distribution is its mean parameter:


$$\mathbb{E}[X_i \mid C_i=k] = \mu_k$$


This serves as the "center" of the cluster in the feature space. The first expectation, $\mathbb{E}[Z_i \mid X_i=x_i]$, outputs a $K$-dimensional vector of probabilities mapping the point to clusters. The second, $\mathbb{E}[X_i \mid C_i=k]$, outputs a $d$-dimensional vector mapping the cluster back to the physical feature space.

**6. Complete-Data Likelihood**
Taking the log of the complete-data likelihood product:


$$\ln p(X, Z) = \ln \left( \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \ln \phi_k + \ln \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$


If $z_{ik}$ were known, this expression would decouple the parameters for each cluster completely. We could maximize the parameters for cluster $k$ independently using only the data points where $z_{ik} = 1$, reducing the problem to standard Gaussian maximum likelihood estimation.

**7 & 8. EM Interpretation and Updates**
The expected complete-data log-likelihood (the $Q$-function) replaces the unobserved $z_{ik}$ with our best probabilistic guess given the data, $\gamma_{ik}$. The E-step is a conditional update because it redistributes the fractional "membership" of every point based on the current model parameters. In the M-step updates, $\gamma_{ik}$ acts as a fractional weight; instead of computing the mean using only points strictly inside the cluster, $\mu_k^{\text{new}}$ is calculated as a weighted average of all data points, scaled by how much they "belong" to cluster $k$.

**9. Interpretation Paragraph**
Gaussian Mixture Clustering is fundamentally an iterative process of conditional updating. We start with a prior probability of cluster membership, $\phi_k$. When a data point $x_i$ is observed, we evaluate how structurally compatible it is with each cluster using the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$. Bayes' theorem is then applied to compute the responsibility $\gamma_{ik}$, representing the updated posterior probability of cluster $k$ conditional on the observation. Because the true cluster identity is a latent variable $Z_i$, this posterior vector perfectly mirrors the mathematical conditional expectation $\mathbb{E}[Z_i \mid X_i=x_i]$. During the M-step, the model incorporates this new evidence, utilizing the posterior probabilities as fractional weights to recalculate the cluster centers and shapes. This cycle repeats, solidifying GMMs as a purely probabilistic clustering mechanism based entirely on the expectations of latent variables.